In [5]:
"""
Check whether the IPTA MDC2 G2D2 CW source is evolving.

Reference: Peters (1964), Phys. Rev. 136, B1224
           Maggiore (2007), Eq. 4.21
"""

import pickle, numpy as np

# =============================================================
# STEP 1: Get observation span from actual TOAs
# =============================================================
print("STEP 1: Loading pulsar TOAs from pickle file")
print("=" * 60)

pkl_path = '/scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/psr_objects/G2D2_IPTA_MDC2_all_pulsars.pkl'

with open(pkl_path, 'rb') as f:
    psrs = pickle.load(f)

print(f"  Loaded {len(psrs)} pulsars")

# enterprise stores TOAs in seconds, convert to MJD
toa_min_mjd = min(np.min(p.toas) for p in psrs) / 86400.0
toa_max_mjd = max(np.max(p.toas) for p in psrs) / 86400.0
T_obs_days  = toa_max_mjd - toa_min_mjd
T_obs_sec   = T_obs_days * 86400.0
T_obs_yr    = T_obs_days / 365.25

print(f"  Earliest TOA across all pulsars: MJD {toa_min_mjd:.4f}")
print(f"  Latest TOA across all pulsars:   MJD {toa_max_mjd:.4f}")
print(f"  Observation span: {toa_max_mjd:.4f} - {toa_min_mjd:.4f} = {T_obs_days:.1f} days")
print(f"  In years: {T_obs_days:.1f} / 365.25 = {T_obs_yr:.2f} yr")
print(f"  In seconds: {T_obs_days:.1f} * 86400 = {T_obs_sec:.4e} s")
print()

# =============================================================
# STEP 2: Injection parameters from MDC2 JSON
# =============================================================
print("STEP 2: MDC2 G2D2 injection parameters (from JSON)")
print("=" * 60)

fGW      = 3.7e-09       # Hz
Mc_solar = 4.3e9          # solar masses

print(f"  fGW      = {fGW:.2e} Hz")
print(f"  Mc       = {Mc_solar:.2e} solar masses")
print()

# =============================================================
# STEP 3: Convert chirp mass to geometrized units
# =============================================================
print("STEP 3: Convert chirp mass to geometrized units (G = c = 1)")
print("=" * 60)

MSUN_SEC = 4.925490947e-6  # 1 Msun in seconds
Mc_sec   = Mc_solar * MSUN_SEC

print(f"  1 solar mass = G * Msun / c^3 = {MSUN_SEC:.9e} seconds")
print(f"  Mc in seconds = {Mc_solar:.2e} * {MSUN_SEC:.6e} = {Mc_sec:.6e} s")
print()

# =============================================================
# STEP 4: Compute frequency derivative (fdot)
#
# Formula: fdot = (96/5) * pi^(8/3) * Mc^(5/3) * fGW^(11/3)
#
# This is the rate at which fGW changes due to GW emission
# for a circular binary at leading (quadrupole) order.
#
# Source: Peters (1964) Phys. Rev. 136, B1224
#         Maggiore (2007) Gravitational Waves, Eq. 4.21
# =============================================================
print("STEP 4: Compute frequency derivative fdot")
print("=" * 60)
print("  Formula: fdot = (96/5) * pi^(8/3) * Mc^(5/3) * fGW^(11/3)")
print()

term1 = 96.0 / 5.0
term2 = np.pi**(8.0/3.0)
term3 = Mc_sec**(5.0/3.0)
term4 = fGW**(11.0/3.0)
fdot  = term1 * term2 * term3 * term4

print(f"  96/5                = {term1:.1f}")
print(f"  pi^(8/3)            = {term2:.6e}")
print(f"  Mc^(5/3)            = ({Mc_sec:.4e})^(5/3) = {term3:.6e}")
print(f"  fGW^(11/3)          = ({fGW:.2e})^(11/3) = {term4:.6e}")
print(f"  fdot = {term1:.1f} * {term2:.4e} * {term3:.4e} * {term4:.4e}")
print(f"       = {fdot:.6e} Hz/s")
print()

# =============================================================
# STEP 5: Characteristic timescale
# =============================================================
print("STEP 5: Characteristic evolution timescale")
print("=" * 60)

tau_c    = fGW / fdot
tau_c_yr = tau_c / (365.25 * 86400.0)

print(f"  tau_c = fGW / fdot = {fGW:.2e} / {fdot:.4e} = {tau_c:.4e} s")
print(f"  In years: {tau_c:.4e} / {365.25*86400:.4e} = {tau_c_yr:.2e} yr")
print(f"  Ratio: tau_c / T_obs = {tau_c_yr:.0f} / {T_obs_yr:.1f} = {tau_c_yr/T_obs_yr:.0f}")
print()

# =============================================================
# STEP 6: Frequency drift over observation
# =============================================================
print("STEP 6: How much does frequency change over T_obs?")
print("=" * 60)

delta_f = fdot * T_obs_sec
df_res  = 1.0 / T_obs_sec

print(f"  delta_f = fdot * T_obs = {fdot:.4e} * {T_obs_sec:.4e} = {delta_f:.4e} Hz")
print(f"  PTA frequency resolution = 1/T_obs = {df_res:.4e} Hz")
print(f"  delta_f / resolution = {delta_f/df_res:.4e}")
print(f"    (frequency drift is {1.0/(delta_f/df_res):.0f}x smaller than what the PTA can resolve)")
print()

# =============================================================
# STEP 7: Phase drift from evolution
#
# If the frequency were constant, the phase would be:
#   Phi(t) = Phi0 + 2*pi*fGW*(t - tref)
#
# With evolution, there is an extra term:
#   Phi(t) = Phi0 + 2*pi*fGW*(t - tref) + pi*fdot*(t - tref)^2
#
# The extra part over the full observation is:
#   delta_Phi = pi * fdot * T_obs^2
#
# If delta_Phi < 1 rad, the constant frequency model fits
# the data and the source is operationally monochromatic.
# =============================================================
print("STEP 7: Phase drift from frequency evolution")
print("=" * 60)

delta_Phi = np.pi * fdot * T_obs_sec**2

print(f"  delta_Phi = pi * fdot * T_obs^2")
print(f"            = {np.pi:.4f} * {fdot:.4e} * ({T_obs_sec:.4e})^2")
print(f"            = {np.pi:.4f} * {fdot:.4e} * {T_obs_sec**2:.4e}")
print(f"            = {delta_Phi:.6f} rad")
print(f"            = {delta_Phi/(2*np.pi):.6f} full cycles")
print()

# =============================================================
# RESULT
# =============================================================
print("=" * 60)
print("RESULT")
print("=" * 60)
print()
if delta_Phi < 1.0:
    print(f"  delta_Phi = {delta_Phi:.4f} rad  <<  1 rad")
    print(f"  The source is monochromatic over this observation span.")
else:
    print(f"  delta_Phi = {delta_Phi:.4f} rad  >  1 rad")
    print(f"  Source evolution may be detectable.")

STEP 1: Loading pulsar TOAs from pickle file
  Loaded 33 pulsars
  Earliest TOA across all pulsars: MJD 52710.6667
  Latest TOA across all pulsars:   MJD 58175.6736
  Observation span: 58175.6736 - 52710.6667 = 5465.0 days
  In years: 5465.0 / 365.25 = 14.96 yr
  In seconds: 5465.0 * 86400 = 4.7218e+08 s

STEP 2: MDC2 G2D2 injection parameters (from JSON)
  fGW      = 3.70e-09 Hz
  Mc       = 4.30e+09 solar masses

STEP 3: Convert chirp mass to geometrized units (G = c = 1)
  1 solar mass = G * Msun / c^3 = 4.925490947e-06 seconds
  Mc in seconds = 4.30e+09 * 4.925491e-06 = 2.117961e+04 s

STEP 4: Compute frequency derivative fdot
  Formula: fdot = (96/5) * pi^(8/3) * Mc^(5/3) * fGW^(11/3)

  96/5                = 19.2
  pi^(8/3)            = 2.117059e+01
  Mc^(5/3)            = (2.1180e+04)^(5/3) = 1.621300e+07
  fGW^(11/3)          = (3.70e-09)^(11/3) = 1.211731e-31
  fdot = 19.2 * 2.1171e+01 * 1.6213e+07 * 1.2117e-31
       = 7.985532e-22 Hz/s

STEP 5: Characteristic evolution times